# Prueba de concepto — Recuperación documental y salvaguarda de abstención

### Asistente conversacional SLM + RAG para la consulta digital del quechua wanka de Junín

**Taller de Proyectos 1 · Universidad Continental · Escuela Académico Profesional de Ingeniería de Sistemas e Informática · Periodo 2026-20 · Grupo 02**

---

Este cuaderno acompaña al **Documento 6 — Prueba de Concepto** y ejecuta, una por una, todas las
mediciones consignadas en él. Cada bloque enuncia primero **qué pregunta responde**, después ejecuta
el experimento y termina con **cómo se lee el resultado**.

#### Qué se pone a prueba

| Hipótesis | Enunciado | Criterio de éxito |
|---|---|---|
| **H1** | Una estrategia de recuperación sobre representaciones vectoriales sitúa el fragmento pertinente entre los cinco primeros resultados. | recall@5 ≥ 0,80 en los subconjuntos A y B |
| **H2** | Existe un umbral de similitud que separa las consultas con respaldo documental de las que carecen de él. | Cero falsos positivos conservando ≥ 70 % de las consultas atendibles |
| **H3** | El codificador multilingüe atiende consultas formuladas en inglés sobre un corpus en español y quechua. | recall@5 ≥ 0,80 en el subconjunto D |

#### Estructura del cuaderno

| Bloque | Contenido | Qué produce |
|---|---|---|
| **1** | Entorno y datos de entrada | Corpus y conjunto de evaluación cargados |
| **2** | Inventario del corpus | Figura 1 y tabla del corpus |
| **3** | Conjunto de evaluación | Figura 2 y composición de los cuatro subconjuntos |
| **4** | Núcleo: normalización, depuración e índices | Funciones comunes a todos los experimentos |
| **5** | Anatomía de una consulta | Traza paso a paso de una recuperación |
| **6** | E1–E4 · estrategias léxicas | Tabla y figura de recuperación |
| **7** | E5 · codificador denso multilingüe | Métricas del modelo denso |
| **8** | Compresión de similitud | Figura comparada de distribuciones |
| **9** | H2 · barrido del umbral de abstención | Punto de operación con cero falsos positivos |
| **10** | E6 · atribución a las decisiones de ingeniería | Efecto de la segmentación y de la depuración |
| **11** | **E7 · traducción EN→ES antes de recuperar** | Contraste de H3 y decisión sobre RF-04 |
| **12** | Prototipo mínimo | Respuesta con procedencia y abstención |
| **13** | Reporte consolidado y decisión | `resultados_cuaderno.json` |

#### Reglas que este cuaderno respeta

1. **Ninguna forma quechua se redacta, se completa ni se corrige.** Todas salen literales del fragmento
   recuperado, con su documento y su página.
2. **Ninguna respuesta se emite sin documento y página.**
3. **El umbral se fija por ausencia de falsos positivos**, nunca por la métrica que da la mejor cifra.
4. **Las cuatro particiones se reportan por separado**; un promedio global ocultaría que el subconjunto
   en inglés falla.
5. **Ninguna cifra se transcribe a mano**: todas las de este cuaderno se calculan al ejecutarlo.

> **Entorno recomendado.** Entorno de ejecución con GPU T4 (`Entorno de ejecución → Cambiar tipo de
> entorno de ejecución → T4 GPU`). Los bloques 1 a 6 y 9 a 10 funcionan también en CPU; los bloques 7
> y 11 descargan modelos y se benefician de la GPU.

---
## Bloque 1 · Entorno y datos de entrada

**Qué hace.** Instala las dependencias y carga los dos archivos que produce la ingesta local:

| Archivo | Contenido |
|---|---|
| `fragmentos_v2.jsonl` | Un objeto por fragmento indexado, con `id`, `documento`, `pagina`, `texto` y `tipo`. |
| `evaluacion_v2.json` | Las consultas de evaluación con su `subconjunto`, su `forma_esperada` y los identificadores de los fragmentos de referencia (`oro`). |

**Por qué se reutilizan y no se reconstruyen aquí.** El corpus se construye en el equipo del proyecto a
partir de los ocho documentos en PDF. Si este cuaderno volviera a construirlo, cualquier diferencia de
resultado podría deberse a la ingesta y no al codificador, que es lo que se quiere medir. La única
variable entre plataformas debe ser cómo se convierte el texto en vectores.

In [ ]:
#@title 1.1 · Instalación de dependencias { display-mode: "form" }
# Ejecutar una sola vez por sesión. Tarda entre uno y tres minutos.
!pip install -q rank_bm25 sentence-transformers sacremoses sentencepiece 2>/dev/null
print("Dependencias instaladas.")

In [ ]:
#@title 1.2 · Importaciones y estilo de las figuras { display-mode: "form" }
import json, re, time, unicodedata, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# Paleta coherente con la del Documento 6
AZUL, VERDE, ROJO, GRIS, NARANJA = "#2E5496", "#4E9A4E", "#B03A2E", "#7F7F7F", "#D68910"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9.5, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold", "figure.autolayout": True,
})

def titulo(t, sub=""):
    # Encabezado uniforme para la salida de texto de cada bloque
    print("=" * 92); print(t.upper())
    if sub: print(sub)
    print("=" * 92)

RESULTADOS = {}      # acumulador global; el bloque 13 lo exporta a JSON
print("Entorno preparado.")

In [ ]:
#@title 1.3 · Carga del corpus y del conjunto de evaluación { display-mode: "form" }
# Tres orígenes posibles, en este orden:
#   a) los archivos ya presentes en /content
#   b) una carpeta de Google Drive montada
#   c) subida manual desde el equipo
RUTA_FRAG, RUTA_EVAL = "fragmentos_v2.jsonl", "evaluacion_v2.json"

if not (os.path.exists(RUTA_FRAG) and os.path.exists(RUTA_EVAL)):
    try:
        from google.colab import files
        print("Seleccione los archivos fragmentos_v2.jsonl y evaluacion_v2.json")
        subidos = files.upload()
    except Exception as e:
        raise SystemExit(f"No fue posible cargar los archivos de entrada: {e}")

FRAG = [json.loads(l) for l in open(RUTA_FRAG, encoding="utf-8") if l.strip()]
EVAL = json.load(open(RUTA_EVAL, encoding="utf-8"))
IDS  = [f["id"] for f in FRAG]
POS  = {d: i for i, d in enumerate(IDS)}

SUBS = ["A_lexico_es", "B_independiente", "D_ingles", "C_fuera_de_cobertura"]
ETIQUETA = {"A_lexico_es": "A · léxico en español",
            "B_independiente": "B · fraseo independiente",
            "D_ingles": "D · consultas en inglés",
            "C_fuera_de_cobertura": "C · fuera de cobertura"}

titulo("datos de entrada cargados")
print(f"Fragmentos indexados ......... {len(FRAG):,}")
print(f"Documentos distintos ......... {len(set(f['documento'] for f in FRAG))}")
print(f"Caracteres en los fragmentos . {sum(len(f['texto']) for f in FRAG):,}")
print(f"Consultas de evaluación ...... {len(EVAL)}")
print()
print("Ejemplo de fragmento:"); print(" ", json.dumps(FRAG[0], ensure_ascii=False))
print("Ejemplo de consulta:");  print(" ", json.dumps(EVAL[0], ensure_ascii=False))

---
## Bloque 2 · Inventario del corpus

**Qué pregunta responde.** ¿Qué material se llegó realmente a incorporar y con qué calidad?

El riesgo R-01 del Documento 1 —corpus insuficiente— es el que puede invalidar el proyecto entero, de
modo que la primera medición no es de desempeño sino de materia prima. Interesan dos magnitudes que se
comportan de forma distinta: **cuánto texto aporta cada documento** y **cuántos fragmentos útiles
produce**. Un diccionario aporta poco texto y muchísimos fragmentos consultables; una gramática aporta
mucho texto y pocos fragmentos autónomos.

In [ ]:
#@title 2.1 · Composición del corpus por documento { display-mode: "form" }
df_doc = (pd.DataFrame(FRAG)
          .assign(caracteres=lambda d: d.texto.str.len())
          .groupby(["documento", "tipo"], as_index=False)
          .agg(fragmentos=("id", "count"), caracteres=("caracteres", "sum"))
          .sort_values("caracteres", ascending=False))
df_doc["% del texto"] = (100 * df_doc.caracteres / df_doc.caracteres.sum()).round(1)
df_doc["% de fragmentos"] = (100 * df_doc.fragmentos / df_doc.fragmentos.sum()).round(1)
df_doc["documento"] = df_doc.documento.str.replace(".pdf", "", regex=False).str.slice(0, 46)

titulo("inventario del corpus indexado")
display(df_doc.reset_index(drop=True))

por_tipo = df_doc.groupby("tipo")[["caracteres", "fragmentos"]].sum()
print("\nPor tipo documental:")
for t, r in por_tipo.iterrows():
    print(f"  {t:<16} {r.caracteres:>8,} caracteres ({100*r.caracteres/por_tipo.caracteres.sum():4.1f} %)"
          f"   ·   {r.fragmentos:>5,} fragmentos ({100*r.fragmentos/por_tipo.fragmentos.sum():4.1f} %)")
RESULTADOS["corpus"] = {"documentos": int(df_doc.documento.nunique()),
                        "fragmentos": len(FRAG),
                        "caracteres": int(df_doc.caracteres.sum()),
                        "por_tipo": por_tipo.to_dict()}

In [ ]:
#@title 2.2 · Figura 1 · Texto aportado frente a fragmentos producidos { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
d = df_doc.sort_values("caracteres")
col = [VERDE if t == "lexicografico" else AZUL for t in d.tipo]

ax[0].barh(d.documento, d.caracteres, color=col)
ax[0].set_title("Texto aportado por documento")
ax[0].set_xlabel("caracteres extraídos")

d2 = df_doc.sort_values("fragmentos")
col2 = [VERDE if t == "lexicografico" else AZUL for t in d2.tipo]
ax[1].barh(d2.documento, d2.fragmentos, color=col2)
ax[1].set_title("Fragmentos indexados por documento")
ax[1].set_xlabel("fragmentos")
ax[1].set_yticklabels([])

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=VERDE, label="material lexicográfico"),
                    Patch(color=AZUL,  label="prosa")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Figura 1. El documento que más texto aporta no es el que más fragmentos útiles produce",
             fontweight="bold", y=1.03)
plt.show()

**Cómo se lee.** Las dos mitades de la figura están deliberadamente desalineadas: el documento que
domina la izquierda no domina la derecha. La gramática aporta la mayor parte del texto, pero es prosa
descriptiva sobre la lengua; el material lexicográfico aporta una fracción del texto y la mayoría de
los fragmentos consultables, que son los que sostienen la historia de usuario HU-03. La conclusión
operativa para el PMV1 es que **ampliar el corpus rinde más por el lado lexicográfico** que por el
lado de la prosa.

---
## Bloque 3 · El conjunto de evaluación

**Qué pregunta responde.** ¿Con qué consultas se mide, y qué mide cada grupo de ellas?

El conjunto tiene cuatro particiones que **se reportan siempre por separado**, porque cada una responde
a una pregunta distinta y un promedio global las volvería indistinguibles:

| Partición | Qué mide | Comportamiento correcto |
|---|---|---|
| **A · léxico en español** | Recuperación en la condición más favorable. | Responder con el fragmento pertinente |
| **B · fraseo independiente** | Sensibilidad a cómo esté redactada la pregunta. | Responder con el fragmento pertinente |
| **D · consultas en inglés** | Requisito RF-04 e historia HU-05. | Responder con el fragmento pertinente |
| **C · fuera de cobertura** | Requisito RF-08: la salvaguarda de abstención. | **Abstenerse** |

La partición C es la que distingue esta prueba de un banco de pruebas convencional: sus consultas
—nociones contemporáneas ajenas al corpus— **no tienen respuesta correcta**. El acierto consiste
exactamente en no responderlas.

**Una honestidad metodológica que conviene declarar aquí.** Las consultas de A, B y D se derivaron
automáticamente de las entradas del propio diccionario que se indexa. Eso garantiza que ninguna forma
quechua fue redactada por el equipo, pero también implica que sus cifras representan el **límite
superior** del desempeño y no su valor en uso real. Es la primera limitación del Documento 6 y la
condición C-6 de su decisión.

In [ ]:
#@title 3.1 · Composición del conjunto de evaluación { display-mode: "form" }
df_eval = pd.DataFrame(EVAL)
comp = (df_eval.groupby("subconjunto")
        .agg(consultas=("id", "count"),
             long_media=("consulta", lambda s: round(s.str.len().mean(), 1)),
             frag_oro_medio=("oro", lambda s: round(np.mean([len(x) for x in s]), 2)))
        .reindex(SUBS))
comp["comportamiento correcto"] = ["responder", "responder", "responder", "abstenerse"]
comp.index = [ETIQUETA[s] for s in comp.index]

titulo("composición del conjunto de evaluación")
display(comp)

print("\nMuestra de consultas por partición:\n")
for s in SUBS:
    ej = [e for e in EVAL if e["subconjunto"] == s][:3]
    print(f"  {ETIQUETA[s]}")
    for e in ej:
        esperada = e.get("forma_esperada") or "— (sin respuesta correcta: debe abstenerse)"
        print(f"    · {e['consulta']!r}  →  {esperada}")
    print()
RESULTADOS["evaluacion"] = {s: int((df_eval.subconjunto == s).sum()) for s in SUBS}

In [ ]:
#@title 3.2 · Figura 2 · Las cuatro particiones y su papel { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))

n = [int((df_eval.subconjunto == s).sum()) for s in SUBS]
colores = [AZUL, AZUL, NARANJA, ROJO]
b = ax[0].bar([ETIQUETA[s].split(" · ")[0] for s in SUBS], n, color=colores)
ax[0].bar_label(b, padding=2)
ax[0].set_title("Consultas por partición")
ax[0].set_ylabel("número de consultas")
ax[0].set_ylim(0, max(n) * 1.18)

resp, abst = sum(n[:3]), n[3]
ax[1].barh(["comportamiento\nesperado"], [resp], color=VERDE, label=f"responder ({resp})")
ax[1].barh(["comportamiento\nesperado"], [abst], left=[resp], color=ROJO, label=f"abstenerse ({abst})")
ax[1].set_title("El desequilibrio que hace inservible la exactitud")
ax[1].set_xlabel("consultas")
ax[1].legend(frameon=False, loc="lower right")
ax[1].grid(False)
ax[1].text(resp / 2, 0, f"{100*resp/(resp+abst):.1f} %", ha="center", va="center", color="white", fontweight="bold")

fig.suptitle("Figura 2. Un sistema que respondiera siempre alcanzaría un 88,7 % de exactitud sin abstenerse jamás",
             fontweight="bold", y=1.05)
plt.show()

**Cómo se lee.** La mitad derecha explica por qué el Documento 6 descarta la exactitud como métrica
principal. Con 220 consultas que deben responderse y 28 que deben rechazarse, un sistema que
respondiera absolutamente todo obtendría cerca de un 89 % de exactitud incumpliendo por completo el
único requisito que esta prueba existe para validar. Por eso H2 se contrasta con la **matriz de
confusión completa** y con el **recuento explícito de falsos positivos**, no con una cifra agregada.

---
## Bloque 4 · El núcleo: normalización, depuración e índices

**Qué pregunta responde.** ¿Qué transformaciones sufre un texto antes de convertirse en un vector?

Este bloque define las funciones que **todos** los experimentos comparten. Que sean comunes es lo que
permite atribuir después cada diferencia de resultado a la estrategia de recuperación y no a un
preprocesamiento distinto.

#### Las dos operaciones que importan

**Normalización.** Pasa a minúsculas, retira los diacríticos y comprime los espacios. Su función es que
«Atuq», «atuq» y «ATUQ» sean el mismo texto, y que la variación ortográfica entre fuentes —el riesgo
R-04 del Documento 1— no impida la coincidencia.

**Depuración de la consulta.** Elimina el fraseo constante que acompaña a toda pregunta: «cómo se
dice», «en quechua wanka», «how do you say». Su justificación es estadística: una palabra que aparece
en las 248 consultas no distingue ninguna de ellas, pero sí aporta similitud a **cualquier** fragmento
que la contenga, incluidos los que no tienen nada que ver con la consulta. Ese aporte espurio es
particularmente dañino para la abstención, porque eleva la puntuación de las consultas que deberían
quedar por debajo del umbral. El bloque 10 mide exactamente cuánto.

#### Los tres índices

| Índice | Qué representa | Para qué sirve |
|---|---|---|
| **BM25** | Coincidencia de términos ponderada por frecuencia y longitud. | Referencia clásica de recuperación léxica |
| **TF-IDF de palabras** (1–2 gramas) | Vector de palabras completas, similitud del coseno. | Precisión en la coincidencia exacta y **similitud acotada entre 0 y 1** |
| **TF-IDF de caracteres** (3–5, `char_wb`) | Vector de secuencias de caracteres. | Tolerancia a la variación ortográfica |

La condición de estar **acotada entre 0 y 1** no es un detalle: BM25 produce una puntuación sin cota
superior, comparable dentro de una consulta pero no entre consultas distintas, y por eso **no admite un
umbral de abstención**. Es la razón por la que la línea base del proyecto no es BM25 pese a su buena
ordenación.

In [ ]:
#@title 4.1 · Normalización, depuración e índices léxicos { display-mode: "form" }
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from rank_bm25 import BM25Okapi

def norm(s: str) -> str:
    s = unicodedata.normalize("NFD", s.lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", s)

RUIDO = re.compile(
    r"\b(en|el|la|del|de|quechua|wanka|huanca|variedad|como|se|dice|que|es|word|for|in|how|do|you|"
    r"say|the|necesito|termino|para|referirme|estoy|buscando|palabra|designa|what|a)\b")

def depurar(q: str) -> str:
    q = RUIDO.sub(" ", q)
    return re.sub(r"[^a-z0-9 ]", " ", q).strip()

TEXTOS = [norm(f["texto"]) for f in FRAG]
tok = lambda s: re.findall(r"[a-z0-9]+", s)

t0 = time.time()
bm25  = BM25Okapi([tok(t) for t in TEXTOS])
vec_w = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True)
M_w   = normalize(vec_w.fit_transform(TEXTOS))
vec_c = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
M_c   = normalize(vec_c.fit_transform(TEXTOS))

titulo("índices construidos", f"{time.time()-t0:.1f} s sobre {len(FRAG):,} fragmentos")
print(f"Vocabulario de palabras ..... {len(vec_w.vocabulary_):,} términos")
print(f"Vocabulario de caracteres ... {len(vec_c.vocabulary_):,} secuencias")
print()
print("Efecto de la depuración sobre tres consultas reales:\n")
for e in [EVAL[0], [x for x in EVAL if x['subconjunto']=='D_ingles'][0],
          [x for x in EVAL if x['subconjunto']=='C_fuera_de_cobertura'][0]]:
    print(f"  original : {e['consulta']}")
    print(f"  depurada : {depurar(norm(e['consulta']))!r}\n")

---
## Bloque 5 · Anatomía de una consulta

**Qué pregunta responde.** ¿Qué ocurre exactamente, paso a paso, cuando alguien pregunta algo?

Antes de medir en masa conviene ver el mecanismo funcionando una sola vez y con todos sus números a la
vista. La estrategia **E4 · híbrida léxica** —la que el proyecto adopta— calcula la puntuación así:

$$S_{E4}(q, f) \;=\; \frac{S_{E2}(q,f) + S_{E3}(q,f)}{2}, \qquad S_{E2}, S_{E3} \in [0,1]$$

y decide con una única regla:

$$\text{si } \max_f S_{E4}(q,f) \ge \tau \;\Rightarrow\; \text{responder citando documento y página}$$
$$\text{si } \max_f S_{E4}(q,f) < \tau \;\Rightarrow\; \text{declarar la ausencia de información y no generar forma alguna}$$

**Por qué se promedian las dos ramas y no se usa solo una.** Una consulta sin ninguna relación con el
corpus no comparte palabras con ningún fragmento, de modo que $S_{E2}$ vale exactamente cero; pero
$S_{E3}$ nunca vale cero, porque siempre hay algún fragmento con el que compartir alguna secuencia de
tres caracteres. Ese ruido de fondo obligaría a subir el umbral. El promedio con $S_{E2}$ arrastra las
coincidencias espurias hacia cero y **ensancha la distancia** entre lo respaldado y lo no respaldado,
que es precisamente la magnitud de la que depende la abstención.

In [ ]:
#@title 5.1 · Función de búsqueda común a los experimentos léxicos { display-mode: "form" }
def puntuar(estrategia: str, consulta: str, depurada: bool = True) -> np.ndarray:
    q = norm(consulta)
    if depurada:
        q = depurar(q)
    if estrategia == "bm25":
        s = np.asarray(bm25.get_scores(tok(q)), dtype=float)
        mx = s.max() if s.max() > 0 else 1.0
        return s / mx                      # normalizado por consulta: sirve para ordenar, no para umbral
    if estrategia == "palabra":
        return (M_w @ normalize(vec_w.transform([q])).T).toarray().ravel()
    if estrategia == "caracter":
        return (M_c @ normalize(vec_c.transform([q])).T).toarray().ravel()
    if estrategia == "hibrido":
        a = (M_w @ normalize(vec_w.transform([q])).T).toarray().ravel()
        b = (M_c @ normalize(vec_c.transform([q])).T).toarray().ravel()
        return 0.5 * a + 0.5 * b
    raise ValueError(estrategia)

def buscar(estrategia: str, consulta: str, k: int = 5, depurada: bool = True):
    s = puntuar(estrategia, consulta, depurada)
    idx = np.argpartition(-s, k)[:k]
    idx = idx[np.argsort(-s[idx])]
    return [(FRAG[i], float(s[i])) for i in idx]

print("Funciones de búsqueda definidas: puntuar() y buscar().")

In [ ]:
#@title 5.2 · Traza de dos consultas: una con respaldo y otra sin él { display-mode: "form" }
CONSULTA_CON_RESPALDO = "\u00bfc\u00f3mo se dice zorro en quechua wanka?"  #@param {type:"string"}
CONSULTA_SIN_RESPALDO = "\u00bfc\u00f3mo se dice criptomoneda en quechua wanka?"  #@param {type:"string"}
TAU = 0.41  #@param {type:"number"}

def traza(consulta):
    q_dep = depurar(norm(consulta))
    s_w = float((M_w @ normalize(vec_w.transform([q_dep])).T).toarray().ravel().max())
    s_c = float((M_c @ normalize(vec_c.transform([q_dep])).T).toarray().ravel().max())
    top = buscar("hibrido", consulta, k=3)
    frag, s_h = top[0]
    print("-" * 92)
    print(f"CONSULTA         {consulta}")
    print(f"  1. depuración  {q_dep!r}")
    print(f"  2. E2 palabras     S = {s_w:.3f}")
    print(f"     E3 caracteres   S = {s_c:.3f}")
    print(f"  3. E4 promedio     S = {s_h:.3f}      (umbral \u03c4 = {TAU})")
    if s_h >= TAU:
        print(f"  4. DECISIÓN    S \u2265 \u03c4  \u2192  RESPONDE citando la procedencia")
        print(f"     fragmento   {frag['texto'][:70]!r}")
        print(f"     procedencia {frag['documento'][:44]}, p\u00e1g. {frag['pagina']}")
    else:
        print(f"  4. DECISIÓN    S < \u03c4  \u2192  SE ABSTIENE")
        print( "     salida      «No hay respaldo documental para esa consulta en el corpus indexado.»")
        print( "                 No se genera ninguna forma ling\u00fc\u00edstica.")
    print("     candidatos considerados:")
    for f, s in top:
        print(f"       {s:.3f}  {f['texto'][:58]:<60} [{f['documento'][:28]}, p. {f['pagina']}]")

titulo("anatom\u00eda de la decisi\u00f3n")
traza(CONSULTA_CON_RESPALDO)
traza(CONSULTA_SIN_RESPALDO)

**Cómo se lee.** La diferencia entre las dos trazas no está en el fragmento devuelto —el sistema
siempre devuelve alguno, porque siempre hay un fragmento menos lejano que los demás— sino en la
**puntuación** de ese fragmento. Ahí reside toda la salvaguarda: el sistema no distingue lo que sabe de
lo que no sabe por el contenido de la respuesta, sino por la distancia a la que se encuentra el mejor
candidato. El bloque 9 determina dónde debe situarse exactamente esa frontera.

---
## Bloque 6 · Experimentos E1 a E4 · estrategias léxicas

**Qué pregunta responde.** ¿Cuál de las estrategias léxicas recupera mejor, y a qué costo de latencia?

Se ejecutan las 248 consultas contra los 3 605 fragmentos con cada estrategia y se registran tres
métricas por partición:

- **recall@1** — proporción de consultas cuyo **primer** resultado es el pertinente.
- **recall@5** — proporción de consultas cuyo fragmento pertinente aparece entre los **cinco primeros**.
  Es la métrica de la hipótesis H1, porque la interfaz muestra varios candidatos.
- **MRR@10** — media del inverso de la posición en que aparece el fragmento pertinente; resume la
  calidad de la ordenación en un solo número.

Ninguna de las tres es una probabilidad ni un umbral: **son tasas de acierto en la ordenación**.

In [ ]:
#@title 6.1 · Ejecución del banco de pruebas léxico { display-mode: "form" }
def evaluar(nombre, fn_puntuar, guardar_detalle=True):
    # fn_puntuar(consulta) -> vector de similitudes sobre FRAG
    detalle, tiempos = [], []
    for e in EVAL:
        t0 = time.perf_counter()
        s = fn_puntuar(e["consulta"])
        idx = np.argpartition(-s, 10)[:10]
        idx = idx[np.argsort(-s[idx])]
        tiempos.append((time.perf_counter() - t0) * 1000)
        oro = set(e["oro"])
        rank = next((i + 1 for i, j in enumerate(idx) if IDS[j] in oro), None)
        detalle.append({"id": e["id"], "sub": e["subconjunto"], "rank": rank,
                        "top1": float(s[idx[0]])})
    filas = {}
    for sub in SUBS:
        d = [x for x in detalle if x["sub"] == sub]
        if not d or sub == "C_fuera_de_cobertura":
            continue
        filas[sub] = {
            "recall@1": round(sum(1 for x in d if x["rank"] == 1) / len(d), 3),
            "recall@5": round(sum(1 for x in d if x["rank"] and x["rank"] <= 5) / len(d), 3),
            "MRR@10":   round(sum(1 / x["rank"] for x in d if x["rank"]) / len(d), 3)}
    return {"nombre": nombre, "metricas": filas,
            "latencia_ms": round(float(np.mean(tiempos)), 2),
            "detalle": detalle if guardar_detalle else None}

ESTRATEGIAS = [("E1 · BM25 léxico", "bm25"), ("E2 · TF-IDF de palabras", "palabra"),
               ("E3 · TF-IDF de caracteres", "caracter"), ("E4 · híbrido léxico", "hibrido")]

EXP = {}
for nombre, clave in ESTRATEGIAS:
    EXP[clave] = evaluar(nombre, lambda c, k=clave: puntuar(k, c, depurada=True))
    print(f"  {nombre:<28} ejecutado  ({EXP[clave]['latencia_ms']} ms por consulta)")

filas = []
for _, clave in ESTRATEGIAS:
    r = EXP[clave]
    filas.append({"Estrategia": r["nombre"],
                  "A · r@1": r["metricas"]["A_lexico_es"]["recall@1"],
                  "A · r@5": r["metricas"]["A_lexico_es"]["recall@5"],
                  "A · MRR": r["metricas"]["A_lexico_es"]["MRR@10"],
                  "B · r@5": r["metricas"]["B_independiente"]["recall@5"],
                  "D · r@5": r["metricas"]["D_ingles"]["recall@5"],
                  "Latencia (ms)": r["latencia_ms"]})
tabla_lex = pd.DataFrame(filas).set_index("Estrategia")

titulo("resultados de recuperación · estrategias léxicas")
display(tabla_lex.style.background_gradient(cmap="Blues", subset=["A · r@5", "B · r@5", "D · r@5"], vmin=0, vmax=1)
        .format("{:.3f}", subset=[c for c in tabla_lex.columns if c != "Latencia (ms)"]))
RESULTADOS["lexico"] = {k: {"nombre": v["nombre"], "metricas": v["metricas"],
                            "latencia_ms": v["latencia_ms"]} for k, v in EXP.items()}

In [ ]:
#@title 6.2 · Figura 3 · Recuperación por estrategia y partición { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9))
etiquetas = [n.split(" · ")[0] for n, _ in ESTRATEGIAS]
x = np.arange(len(etiquetas)); w = 0.26

for i, (sub, col) in enumerate([("A_lexico_es", AZUL), ("B_independiente", VERDE), ("D_ingles", NARANJA)]):
    v = [EXP[c]["metricas"][sub]["recall@5"] for _, c in ESTRATEGIAS]
    b = ax[0].bar(x + (i - 1) * w, v, w, label=ETIQUETA[sub], color=col)
    ax[0].bar_label(b, fmt="%.2f", fontsize=7, padding=1)
ax[0].axhline(0.80, color=ROJO, ls="--", lw=1.2)
ax[0].text(len(x) - 0.4, 0.82, "criterio de éxito 0,80", color=ROJO, fontsize=8, ha="right")
ax[0].set_xticks(x); ax[0].set_xticklabels(etiquetas)
ax[0].set_ylim(0, 1.15); ax[0].set_ylabel("recall@5")
ax[0].set_title("El español se resuelve; el inglés no")
ax[0].legend(frameon=False, fontsize=8, ncol=1, loc="upper left")

lat = [EXP[c]["latencia_ms"] for _, c in ESTRATEGIAS]
b = ax[1].bar(etiquetas, lat, color=GRIS)
ax[1].bar_label(b, fmt="%.1f ms", fontsize=8, padding=2)
ax[1].axhline(100, color=ROJO, ls="--", lw=1.2)
ax[1].text(len(x) - 0.4, 104, "RNF-02: 100 ms", color=ROJO, fontsize=8, ha="right")
ax[1].set_yscale("log"); ax[1].set_ylabel("ms por consulta (escala logarítmica)")
ax[1].set_title("La latencia no es una restricción a esta escala")

fig.suptitle("Figura 3. Recuperación y latencia de las cuatro estrategias léxicas", fontweight="bold", y=1.04)
plt.show()

**Cómo se lee.** Tres lecturas y una advertencia.

1. **En español el problema está resuelto**: las estrategias con vocabulario de palabras sitúan el
   fragmento pertinente entre los cinco primeros en la práctica totalidad de las consultas, y el
   resultado se mantiene al cambiar el fraseo (partición B), de modo que no es un artefacto de cómo
   estén redactadas las preguntas.
2. **En inglés ninguna estrategia se acerca al criterio.** La única que rescata algo es la de
   n-gramas de caracteres, y lo hace por coincidencia parcial de raíces latinas, no por comprensión.
3. **La latencia sobra por dos órdenes de magnitud** frente al límite de 100 ms del requisito RNF-02.
   Con este tamaño de corpus, el costo de la recuperación no condiciona ninguna decisión de diseño.

**La advertencia** es sobre E1. BM25 ordena tan bien como las demás, pero su puntuación no está acotada:
para poder compararla entre consultas hay que normalizarla por consulta, y entonces el valor máximo es
siempre 1 tanto si el fragmento es pertinente como si no. Eso la vuelve **inservible como umbral de
abstención**, que es la razón por la que la línea base del proyecto es E4 y no E1.

---
## Bloque 7 · Experimento E5 · codificador denso multilingüe

**Qué pregunta responde.** ¿Resuelve el problema un codificador que representa el **significado** en
lugar de las palabras?

Es la pregunta central del cuaderno, porque el Documento 5 había seleccionado un codificador denso
multilingüe —la familia **multilingual-E5**— por su capacidad translingüe declarada, y por tanto
esperaba que fuese esta estrategia la que resolviera el subconjunto en inglés.

#### Cómo funciona, en una frase

Un codificador denso proyecta cada texto en un vector de varios cientos de dimensiones aprendido a
partir de millones de pares pregunta–pasaje, de modo que dos textos con significado próximo quedan
próximos en ese espacio aunque no compartan ninguna palabra. La similitud se mide con el coseno del
ángulo entre ambos vectores.

#### Dos detalles de implementación que invalidan el experimento si se omiten

| Detalle | Por qué importa |
|---|---|
| **Prefijos `query:` y `passage:`** | La familia E5 se entrena de forma **asimétrica**: aprende a acercar el vector de una pregunta al de un pasaje que la responde, y distingue ambos papeles por ese prefijo. Codificar sin él degrada el resultado por una razón ajena al corpus, y llevaría a concluir equivocadamente que el modelo no sirve. |
| **Normalizar los vectores** | Con vectores de norma uno, el producto escalar **es** el coseno, de modo que toda la matriz de similitudes se obtiene con un único producto de matrices. |

#### Por qué este experimento se ejecuta en Google Colab

No es una decisión de diseño sino de infraestructura: el entorno de trabajo local del proyecto no tiene
acceso al repositorio desde el que se descargan los pesos del modelo. Colab lo resuelve y además aporta
una unidad de procesamiento gráfico. La comparabilidad se asegura porque este cuaderno recibe **los
mismos dos archivos de entrada**, aplica **la misma depuración** y calcula **las mismas métricas**: lo
único que cambia es cómo se convierte el texto en vectores.

In [ ]:
#@title 7.1 · Codificación del corpus con multilingual-E5 { display-mode: "form" }
MODELO = "intfloat/multilingual-e5-small"  #@param ["intfloat/multilingual-e5-small", "intfloat/multilingual-e5-base"]
LOTE = 64  #@param {type:"integer"}

from sentence_transformers import SentenceTransformer
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
titulo(f"codificación densa · {MODELO}", f"dispositivo: {dispositivo}")

modelo = SentenceTransformer(MODELO, device=dispositivo)
print(f"Parámetros del modelo ........ {sum(p.numel() for p in modelo.parameters())/1e6:.0f} M")
print(f"Dimensión de los vectores .... {modelo.get_sentence_embedding_dimension()}")

t0 = time.time()
E_doc = modelo.encode(["passage: " + f["texto"] for f in FRAG],
                      batch_size=LOTE, normalize_embeddings=True,
                      show_progress_bar=True, convert_to_numpy=True)
t_corpus = time.time() - t0
print(f"\nCorpus codificado en {t_corpus/60:.1f} min  ->  matriz {E_doc.shape}")

t0 = time.time()
E_con = modelo.encode(["query: " + depurar(norm(e["consulta"])) for e in EVAL],
                      batch_size=LOTE, normalize_embeddings=True,
                      show_progress_bar=True, convert_to_numpy=True)
print(f"Consultas codificadas en {time.time()-t0:.1f} s  ->  matriz {E_con.shape}")

S_densa = E_con @ E_doc.T          # vectores normalizados: cada valor es el coseno
print(f"Matriz de similitudes: {S_densa.shape[0]} consultas x {S_densa.shape[1]} fragmentos")

In [ ]:
#@title 7.2 · Métricas del codificador denso { display-mode: "form" }
POSICION_EVAL = {e["id"]: i for i, e in enumerate(EVAL)}

def puntuar_denso(consulta_id):
    return S_densa[POSICION_EVAL[consulta_id]]

detalle_d = []
for e in EVAL:
    s = puntuar_denso(e["id"])
    idx = np.argsort(-s)[:10]
    oro = set(e["oro"])
    rank = next((i + 1 for i, j in enumerate(idx) if IDS[j] in oro), None)
    detalle_d.append({"id": e["id"], "sub": e["subconjunto"], "rank": rank, "top1": float(s[idx[0]])})

EXP["denso"] = {"nombre": f"E5 · {MODELO.split('/')[-1]}", "detalle": detalle_d,
                "metricas": {}, "latencia_ms": None}
for sub in SUBS:
    d = [x for x in detalle_d if x["sub"] == sub]
    if sub == "C_fuera_de_cobertura":
        continue
    EXP["denso"]["metricas"][sub] = {
        "recall@1": round(sum(1 for x in d if x["rank"] == 1) / len(d), 3),
        "recall@5": round(sum(1 for x in d if x["rank"] and x["rank"] <= 5) / len(d), 3),
        "MRR@10":   round(sum(1 / x["rank"] for x in d if x["rank"]) / len(d), 3)}

comparacion = tabla_lex.copy()
m = EXP["denso"]["metricas"]
comparacion.loc[EXP["denso"]["nombre"]] = [m["A_lexico_es"]["recall@1"], m["A_lexico_es"]["recall@5"],
                                           m["A_lexico_es"]["MRR@10"], m["B_independiente"]["recall@5"],
                                           m["D_ingles"]["recall@5"], np.nan]
titulo("denso frente a léxico")
display(comparacion.style.background_gradient(cmap="Blues", subset=["A · r@5", "B · r@5", "D · r@5"], vmin=0, vmax=1)
        .format("{:.3f}", subset=[c for c in comparacion.columns if c != "Latencia (ms)"], na_rep="—"))
RESULTADOS["denso"] = {"modelo": MODELO, "metricas": m, "minutos_codificacion_corpus": round(t_corpus/60, 2)}

**Cómo se lee.** El resultado contradice la expectativa con la que se planteó el experimento, y esa
contradicción es el hallazgo más valioso de la prueba de concepto.

En español el codificador denso queda muy por debajo de las estrategias léxicas, y en inglés —el
subconjunto que debía justificar su elección— tampoco alcanza el criterio, aunque supere ligeramente a
las léxicas. **La explicación está en la naturaleza del corpus, no en el modelo.** Dos terceras partes
de los fragmentos son entradas de diccionario de tres o cuatro palabras, del tipo `ZORRO: Atuq.`. Un
codificador denso está entrenado para capturar el significado de oraciones y pasajes; ante textos tan
breves dispone de muy poca señal, mientras que la coincidencia léxica exacta es, en ese caso concreto,
casi toda la información disponible.

> **Si dispone de tiempo de ejecución**, repita el bloque 7.1 seleccionando la variante `base` en el
> menú del formulario y vuelva a ejecutar 7.2. El Documento 6 registra ese contraste: triplicar los
> parámetros **mejora el inglés y empeora el español**, de modo que el tamaño del codificador no
> resuelve el problema sino que lo desplaza.

---
## Bloque 8 · La compresión de similitud

**Qué pregunta responde.** ¿Por qué el codificador denso no admite un umbral de abstención absoluto?

Aquí el cuaderno deja de medir *ordenación* y empieza a medir *escala*, que es la propiedad de la que
depende la salvaguarda. Para que un umbral funcione no basta con que el sistema ordene bien: hace falta
que las consultas **con** respaldo documental y las consultas **sin** respaldo caigan en rangos de
puntuación **separados**. Esa separación —y no la similitud absoluta— es lo que hace implementable la
abstención.

In [ ]:
#@title 8.1 · Figura 4 · Distribución de la similitud con y sin respaldo { display-mode: "form" }
def poblaciones(detalle):
    con = [x["top1"] for x in detalle if x["sub"] in ("A_lexico_es", "B_independiente")]
    sin = [x["top1"] for x in detalle if x["sub"] == "C_fuera_de_cobertura"]
    return np.array(con), np.array(sin)

con_l, sin_l = poblaciones(EXP["hibrido"]["detalle"])
con_d, sin_d = poblaciones(EXP["denso"]["detalle"])

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9), sharey=True)
for a, (con, sin, tit) in zip(ax, [(con_l, sin_l, "E4 · híbrido léxico"),
                                   (con_d, sin_d, EXP["denso"]["nombre"])]):
    a.hist(con, bins=28, color=VERDE, alpha=0.75, label="con respaldo (A + B)")
    a.hist(sin, bins=28, color=ROJO, alpha=0.75, label="sin respaldo (C)")
    a.axvline(con.mean(), color=VERDE, ls="--", lw=1.2)
    a.axvline(sin.mean(), color=ROJO,  ls="--", lw=1.2)
    sep = con.mean() - sin.mean()
    a.set_title(f"{tit}\nseparación de medias = {sep:.3f}")
    a.set_xlabel("similitud del mejor fragmento")
    a.legend(frameon=False, fontsize=8)
ax[0].set_ylabel("consultas")

fig.suptitle("Figura 4. Lo que hace posible la abstención no es la similitud alta, sino la distancia entre las dos poblaciones",
             fontweight="bold", y=1.06)
plt.show()

resumen_sep = pd.DataFrame({
    "E4 · híbrido léxico": [con_l.mean(), sin_l.mean(), con_l.mean() - sin_l.mean(), sin_l.max(), con_l.min()],
    EXP["denso"]["nombre"]: [con_d.mean(), sin_d.mean(), con_d.mean() - sin_d.mean(), sin_d.max(), con_d.min()]},
    index=["similitud media con respaldo", "similitud media sin respaldo", "separación de medias",
           "máximo sin respaldo (el falso positivo más peligroso)", "mínimo con respaldo"]).round(3)
display(resumen_sep)
RESULTADOS["separacion"] = resumen_sep.to_dict()

**Cómo se lee.** Las dos mitades de la figura son la misma medición sobre las mismas consultas y
cuentan historias opuestas.

En la estrategia léxica las dos poblaciones ocupan **zonas distintas** del eje: existe un espacio
intermedio por donde puede pasar una frontera. En la estrategia densa ambas poblaciones se apilan en una
banda estrecha y alta —próxima a 0,85— y se solapan casi por completo. **Una similitud de 0,84 no
significa allí que el fragmento sea pertinente**; significa que el codificador sitúa casi todo el corpus
a una distancia parecida de casi cualquier consulta.

De ahí se siguen dos consecuencias que el Documento 6 recoge como condiciones C-4 y C-5:

1. Los umbrales **no son transferibles entre estrategias**: pertenecen a escalas que se comportan de
   manera distinta, de modo que cambiar de codificador obliga a recalibrar por completo.
2. Si el PMV2 conserva el codificador denso para el dispositivo móvil, la decisión de abstenerse **no
   puede apoyarse en el valor absoluto de la similitud**, sino en la diferencia entre el primer y el
   segundo resultado o en una calibración explícita.

---
## Bloque 9 · Hipótesis H2 · el umbral de abstención

**Qué pregunta responde.** ¿Dónde debe situarse la frontera entre responder y abstenerse?

Este es el bloque decisivo del proyecto. Se recorre el umbral en incrementos de 0,01 y en cada punto se
cuenta la matriz de confusión completa, tratando cada consulta como una decisión binaria:

| | El corpus **sí** respalda (A + B) | El corpus **no** respalda (C) |
|---|---|---|
| **El sistema responde** | verdadero positivo · correcto | **falso positivo · el error irreversible** |
| **El sistema se abstiene** | falso negativo · consulta legítima no atendida | verdadero negativo · correcto |

**La asimetría de los dos errores es el fundamento de todo el proyecto.** Un falso negativo deja al
usuario donde estaba y se corrige ampliando el corpus. Un falso positivo introduce en circulación una
forma lingüística que nadie documentó, sobre una lengua seriamente en peligro, y esa forma puede ser
citada, copiada y reutilizada sin que quede rastro de su origen. La medida F1 pondera ambos errores por
igual; el problema real no. Por eso el punto de operación se elige por **el primer umbral sin falsos
positivos**, y no por el que maximiza F1.

In [ ]:
#@title 9.1 · Barrido del umbral por estrategia { display-mode: "form" }
def barrido(detalle, lo=0.02, hi=0.99, paso=0.01):
    con, sin = poblaciones(detalle)
    filas = []
    for u in np.arange(lo, hi + 1e-9, paso):
        vp = int((con >= u).sum()); fn = len(con) - vp
        fp = int((sin >= u).sum()); vn = len(sin) - fp
        prec = vp / (vp + fp) if vp + fp else 0.0
        rec  = vp / (vp + fn) if vp + fn else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        filas.append({"umbral": round(float(u), 2), "VP": vp, "FP": fp, "VN": vn, "FN": fn,
                      "precision": round(prec, 4), "recall": round(rec, 4), "F1": round(f1, 4),
                      "abstencion_correcta": round(vn / len(sin), 4)})
    df = pd.DataFrame(filas)
    sin_fp = df[df.FP == 0]
    # si ninguna posición del barrido elimina los falsos positivos, se devuelve el umbral más alto
    elegido = sin_fp.iloc[0] if len(sin_fp) else df.iloc[-1]
    return {"curva": df,
            "sin_fp": elegido.to_dict(),
            "mejor_f1": df.loc[df.F1.idxmax()].to_dict()}

BARRIDOS = {}
for clave, etiqueta in [("palabra", "E2 · TF-IDF de palabras"), ("caracter", "E3 · TF-IDF de caracteres"),
                        ("hibrido", "E4 · híbrido léxico"), ("denso", EXP["denso"]["nombre"])]:
    BARRIDOS[clave] = barrido(EXP[clave]["detalle"])

filas = []
for clave, etiqueta in [("palabra", "E2 · TF-IDF de palabras"), ("caracter", "E3 · TF-IDF de caracteres"),
                        ("hibrido", "E4 · híbrido léxico"), ("denso", EXP["denso"]["nombre"])]:
    r = BARRIDOS[clave]["sin_fp"]
    filas.append({"Estrategia": etiqueta, "Umbral sin FP": r["umbral"], "VP": int(r["VP"]),
                  "FP": int(r["FP"]), "VN": int(r["VN"]), "FN": int(r["FN"]),
                  "Recall": r["recall"], "F1": r["F1"],
                  "Cumple ≥ 0,70": "sí" if r["recall"] >= 0.70 else "no"})
mf = BARRIDOS["hibrido"]["mejor_f1"]
filas.append({"Estrategia": "E4 · punto de mayor F1 (descartado)", "Umbral sin FP": mf["umbral"],
              "VP": int(mf["VP"]), "FP": int(mf["FP"]), "VN": int(mf["VN"]), "FN": int(mf["FN"]),
              "Recall": mf["recall"], "F1": mf["F1"], "Cumple ≥ 0,70": "descartado: 1 FP" if mf["FP"] else "—"})

tabla_umbral = pd.DataFrame(filas).set_index("Estrategia")
titulo("punto de operación sin falsos positivos")
display(tabla_umbral)
RESULTADOS["umbral"] = tabla_umbral.reset_index().to_dict("records")

TAU_ELEGIDO = float(BARRIDOS["hibrido"]["sin_fp"]["umbral"])
print(f"\nUmbral adoptado para la estrategia híbrida léxica:  τ = {TAU_ELEGIDO}")

In [ ]:
#@title 9.2 · Figura 5 · Recorrido completo de la decisión { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=True)
for a, clave, tit in [(ax[0], "hibrido", "E4 · híbrido léxico"),
                      (ax[1], "denso", EXP["denso"]["nombre"])]:
    c = BARRIDOS[clave]["curva"]
    a.plot(c.umbral, c.F1, color=AZUL, lw=1.8, label="F1 de la decisión de responder")
    a.plot(c.umbral, c.recall, color=VERDE, lw=1.8, label="recall (consultas con respaldo atendidas)")
    a.plot(c.umbral, c.abstencion_correcta, color=ROJO, lw=1.8, label="abstención correcta (partición C)")
    u = BARRIDOS[clave]["sin_fp"]["umbral"]
    a.axvline(u, color=ROJO, ls="--", lw=1.3)
    a.annotate(f"τ = {u}\nprimer umbral\nsin falsos positivos", xy=(u, 0.45),
               xytext=(6, 0), textcoords="offset points", fontsize=8, color=ROJO, fontweight="bold")
    if clave == "hibrido":
        mf = BARRIDOS[clave]["mejor_f1"]["umbral"]
        a.axvline(mf, color=GRIS, ls=":", lw=1.2)
        a.annotate(f"mayor F1 ({mf})\n1 falso positivo", xy=(mf, 0.14), xytext=(-96, 0),
                   textcoords="offset points", fontsize=8, color=GRIS)
    a.set_title(tit); a.set_xlabel("umbral de similitud"); a.set_xlim(0, 1)
ax[0].set_ylabel("valor de la métrica")
ax[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("Figura 5. El punto de operación no es el de mayor F1, sino el primero sin falsos positivos",
             fontweight="bold", y=1.05)
plt.show()

**Cómo se lee.** La figura es el recorrido de una decisión. A medida que el umbral sube, el sistema se
vuelve más exigente: la curva verde baja porque atiende menos consultas legítimas, y la roja sube porque
rechaza más consultas sin respaldo. El punto en que la roja alcanza el valor uno es el primer umbral que
no comete ningún falso positivo, y es el que el proyecto adopta.

La comparación entre las dos mitades resume la elección de estrategia del PMV1. En la léxica, la
ausencia de falsos positivos se consigue **conservando la mayor parte de la cobertura**. En la densa, se
consigue también, pero solo sacrificando la mayoría de las respuestas válidas: la salvaguarda se obtiene
al precio de inutilizar el sistema.

Queda un punto por precisar sobre la naturaleza de τ. **No es una probabilidad ni una confianza del
modelo**: es una puntuación de similitud acotada entre 0 y 1, calibrada sobre un corpus, una
segmentación y un conjunto de consultas concretos. De ahí que el Documento 6 lo trate como **parámetro
versionado en configuración** y no como constante del código, y exija recalibrarlo con este mismo arnés
cada vez que cambie cualquiera de esos tres elementos.

---
## Bloque 10 · Experimento E6 · atribución del resultado a las decisiones de ingeniería

**Qué pregunta responde.** ¿Cuánto del resultado se debe a la estrategia de recuperación y cuánto a dos
decisiones que suelen darse por supuestas?

Las dos decisiones son la **segmentación del corpus** y la **depuración de la consulta**. Ninguna de las
dos la resuelve por defecto una biblioteca de recuperación: hay que tomarlas explícitamente.

**Segmentación.** Lo habitual es trocear todo el corpus en bloques de longitud uniforme. Sobre material
lexicográfico eso mezcla decenas de entradas sin relación entre sí en un mismo fragmento: la entrada
buscada queda diluida entre otras cincuenta y su similitud cae hasta confundirse con el ruido. La
alternativa adoptada es segmentar **por tipo documental**: una entrada por fragmento en el material
lexicográfico, bloques por límite de párrafo en la prosa.

**Depuración.** Ya justificada en el bloque 4. Aquí se mide su efecto.

Este bloque reconstruye la configuración descartada —vuelve a unir el material lexicográfico y lo
trocea en bloques uniformes— y la evalúa con la misma estrategia y el mismo conjunto de consultas, de
modo que la diferencia solo pueda atribuirse a la segmentación.

In [ ]:
#@title 10.1 · Reconstrucción de la segmentación uniforme { display-mode: "form" }
TAMANO_BLOQUE = 650  #@param {type:"integer"}

bloques, contiene = [], []
por_doc = {}
for f in FRAG:
    por_doc.setdefault((f["documento"], f["tipo"]), []).append(f)

for (doc, tipo), lista in por_doc.items():
    if tipo != "lexicografico":
        for f in lista:                       # la prosa ya venía en bloques: se conserva igual
            bloques.append(f["texto"]); contiene.append({f["id"]})
        continue
    acumulado, ids_acum = "", set()
    for f in lista:
        if len(acumulado) + len(f["texto"]) > TAMANO_BLOQUE and acumulado:
            bloques.append(acumulado); contiene.append(ids_acum)
            acumulado, ids_acum = "", set()
        acumulado += (" " if acumulado else "") + f["texto"]
        ids_acum.add(f["id"])
    if acumulado:
        bloques.append(acumulado); contiene.append(ids_acum)

TEXTOS_U = [norm(b) for b in bloques]
vec_wu = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True)
M_wu = normalize(vec_wu.fit_transform(TEXTOS_U))
vec_cu = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
M_cu = normalize(vec_cu.fit_transform(TEXTOS_U))

titulo("segmentación uniforme reconstruida")
print(f"Fragmentos con segmentación por entrada .... {len(FRAG):,}")
print(f"Bloques con segmentación uniforme .......... {len(bloques):,}")
print(f"Entradas léxicas por bloque (media) ........ {np.mean([len(c) for c in contiene]):.1f}")
print("\nEjemplo de bloque uniforme (primeros 300 caracteres):")
print(" ", bloques[0][:300], "...")

In [ ]:
#@title 10.2 · Efecto de la segmentación y de la depuración { display-mode: "form" }
def evaluar_uniforme(depurada=True):
    detalle = []
    for e in EVAL:
        q = norm(e["consulta"])
        if depurada:
            q = depurar(q)
        a = (M_wu @ normalize(vec_wu.transform([q])).T).toarray().ravel()
        b = (M_cu @ normalize(vec_cu.transform([q])).T).toarray().ravel()
        s = 0.5 * a + 0.5 * b
        idx = np.argsort(-s)[:10]
        oro = set(e["oro"])
        rank = next((i + 1 for i, j in enumerate(idx) if contiene[j] & oro), None)
        detalle.append({"id": e["id"], "sub": e["subconjunto"], "rank": rank, "top1": float(s[idx[0]])})
    return detalle

def evaluar_hibrido_sin_depurar():
    detalle = []
    for e in EVAL:
        s = puntuar("hibrido", e["consulta"], depurada=False)
        idx = np.argsort(-s)[:10]
        oro = set(e["oro"])
        rank = next((i + 1 for i, j in enumerate(idx) if IDS[j] in oro), None)
        detalle.append({"id": e["id"], "sub": e["subconjunto"], "rank": rank, "top1": float(s[idx[0]])})
    return detalle

CONFIG = {
    "Uniforme de 650 caracteres · consulta depurada": evaluar_uniforme(depurada=True),
    "Por entrada · consulta SIN depurar":             evaluar_hibrido_sin_depurar(),
    "Por entrada · consulta depurada (adoptada)":     EXP["hibrido"]["detalle"],
}

filas = []
for nombre, det in CONFIG.items():
    a = [x for x in det if x["sub"] == "A_lexico_es"]
    r5 = sum(1 for x in a if x["rank"] and x["rank"] <= 5) / len(a)
    b = barrido(det)
    s = b["sin_fp"]
    filas.append({"Configuración": nombre, "A · r@5": round(r5, 3),
                  "Umbral sin FP": s["umbral"], "Recall en ese umbral": s["recall"], "F1": s["F1"]})
tabla_ablacion = pd.DataFrame(filas).set_index("Configuración")

titulo("atribución del resultado a las decisiones de ingeniería")
display(tabla_ablacion)
RESULTADOS["ablacion"] = tabla_ablacion.reset_index().to_dict("records")

fig, ax = plt.subplots(figsize=(9.5, 3.4))
x = np.arange(len(tabla_ablacion)); w = 0.36
b1 = ax.bar(x - w/2, tabla_ablacion["Umbral sin FP"], w, color=AZUL, label="umbral utilizable (sin falsos positivos)")
b2 = ax.bar(x + w/2, tabla_ablacion["Recall en ese umbral"], w, color=VERDE, label="consultas atendidas en ese umbral")
ax.bar_label(b1, fmt="%.2f", fontsize=8); ax.bar_label(b2, fmt="%.3f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([t.replace(" · ", "\n") for t in tabla_ablacion.index], fontsize=8)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("Figura 6. Dos decisiones de ingeniería, no la estrategia, explican la mayor parte de la ganancia")
plt.show()

**Cómo se lee.** El resultado es el hallazgo de ingeniería de mayor efecto de toda la prueba, y no
tiene que ver con qué algoritmo de recuperación se elija.

Con el diccionario troceado en bloques uniformes, la ordenación apenas se resiente —el fragmento sigue
apareciendo entre los primeros—, pero **la puntuación se hunde**, porque la entrada buscada representa
una fracción diminuta del bloque que la contiene. Y como la puntuación es lo que sostiene el umbral, la
ventana utilizable se estrecha y la cobertura en el punto de operación cae aproximadamente a la mitad.
La depuración de la consulta actúa sobre el otro extremo: sin ella, el fraseo constante regala similitud
a **todas** las consultas, incluidas las que deberían quedar por debajo del umbral.

La consecuencia para el PMV1 es concreta y quedó registrada como condición C-3: la segmentación debe ser
**por tipo documental**, con el tipo almacenado como campo obligatorio del índice, y la depuración de la
consulta debe formar parte del núcleo de dominio, antes de la codificación.

---
## Bloque 11 · Experimento E7 · traducción de la consulta antes de recuperar

**Qué pregunta responde.** ¿Se puede resolver la consulta en inglés sin tocar el corpus ni el
codificador?

Es el requisito que la prueba de concepto declara **incumplido**: ni las estrategias léxicas ni la densa
alcanzan el criterio en el subconjunto D, y el bloque 7 mostró que agrandar el codificador tampoco lo
resuelve. Quedan dos caminos:

| Alternativa | En qué consiste | Costo |
|---|---|---|
| **Traducir la consulta** | Llevar la pregunta del inglés al español **antes** de recuperar, y buscar con las dos versiones. | Un paso adicional por consulta; no altera el corpus |
| **Indexar una glosa** | Añadir al índice una traducción al inglés de cada entrada léxica. | Modifica el corpus indexado y multiplica su tamaño |

Se ensaya la primera, que es la recomendada en el Documento 6 (condición C-1), porque no altera el
material documental.

#### Las cuatro reglas que la traducción debe respetar

1. **Se traduce solo la consulta.** Nunca el corpus, nunca la respuesta.
2. **La forma quechua sale siempre literal del fragmento recuperado.** El traductor no toca quechua en
   ningún punto del flujo; si lo hiciera, se rompería la salvaguarda central del proyecto.
3. **Se recupera con la consulta original y con la traducida**, y se conserva la mejor puntuación de
   cada fragmento. Así una traducción defectuosa degrada el resultado, pero no lo anula.
4. **La traducción se registra en la trazabilidad**, para que la persona usuaria pueda ver con qué
   términos se buscó realmente.

> En el sistema final la traducción la realiza el propio modelo de lenguaje local con una instrucción
> acotada, sin dependencia de servicios externos. Aquí se emplea un modelo de traducción compacto y
> abierto porque el objetivo de este bloque es **medir la ganancia del procedimiento**, no seleccionar
> el traductor definitivo.

In [ ]:
#@title 11.1 · Traducción de las consultas del subconjunto D { display-mode: "form" }
MODELO_TRAD = "Helsinki-NLP/opus-mt-en-es"  #@param ["Helsinki-NLP/opus-mt-en-es", "Helsinki-NLP/opus-mt-mul-en"]

# Se carga el modelo de traducción directamente, sin la interfaz de alto nivel `pipeline`:
# la tarea genérica "translation" no está registrada en todas las versiones de transformers.
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

try:
    dispositivo
except NameError:
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

tok_tr = AutoTokenizer.from_pretrained(MODELO_TRAD)
mod_tr = AutoModelForSeq2SeqLM.from_pretrained(MODELO_TRAD).to(dispositivo).eval()

def traducir_en_es(textos, lote=16, max_tokens=64):
    # Devuelve una cadena si recibe una cadena, y una lista si recibe una lista.
    uno = isinstance(textos, str)
    xs = [textos] if uno else list(textos)
    salida = []
    for i in range(0, len(xs), lote):
        ent = tok_tr(xs[i:i + lote], return_tensors="pt", padding=True,
                     truncation=True, max_length=128).to(dispositivo)
        with torch.no_grad():
            gen = mod_tr.generate(**ent, max_new_tokens=max_tokens, num_beams=4)
        salida += tok_tr.batch_decode(gen, skip_special_tokens=True)
    return salida[0] if uno else salida

D = [e for e in EVAL if e["subconjunto"] == "D_ingles"]
t0 = time.time()
traducciones = traducir_en_es([e["consulta"] for e in D])
t_trad = (time.time() - t0) / len(D) * 1000
TRAD = {e["id"]: t for e, t in zip(D, traducciones)}

titulo("traducción de las 40 consultas en inglés",
       f"{MODELO_TRAD} · {dispositivo} · {t_trad:.0f} ms por consulta")
for e in D[:8]:
    print(f"  EN  {e['consulta']}")
    print(f"  ES  {TRAD[e['id']]}")
    print(f"      depurada: {depurar(norm(TRAD[e['id']]))!r}   ·   se espera recuperar: {e['forma_esperada']}\n")

In [ ]:
#@title 11.2 · Recuperación con la consulta original y con la traducida { display-mode: "form" }
def evaluar_D(usar_traduccion):
    filas = []
    for e in D:
        s = puntuar("hibrido", e["consulta"], depurada=True)
        if usar_traduccion:
            s = np.maximum(s, puntuar("hibrido", TRAD[e["id"]], depurada=True))   # se conserva la mejor
        idx = np.argsort(-s)[:10]
        oro = set(e["oro"])
        rank = next((i + 1 for i, j in enumerate(idx) if IDS[j] in oro), None)
        filas.append({"id": e["id"], "rank": rank, "top1": float(s[idx[0]])})
    return filas

antes, despues = evaluar_D(False), evaluar_D(True)

def resumen_D(filas, tau):
    r1 = sum(1 for x in filas if x["rank"] == 1) / len(filas)
    r5 = sum(1 for x in filas if x["rank"] and x["rank"] <= 5) / len(filas)
    mrr = sum(1 / x["rank"] for x in filas if x["rank"]) / len(filas)
    atendidas = sum(1 for x in filas if x["top1"] >= tau)
    correctas = sum(1 for x in filas if x["top1"] >= tau and x["rank"] and x["rank"] <= 5)
    return {"recall@1": round(r1, 3), "recall@5": round(r5, 3), "MRR@10": round(mrr, 3),
            "similitud media": round(float(np.mean([x["top1"] for x in filas])), 3),
            f"atendidas con τ={tau}": atendidas,
            f"atendidas y correctas": correctas}

tabla_D = pd.DataFrame({
    "Sin traducción (consulta en inglés)": resumen_D(antes, TAU_ELEGIDO),
    "Con traducción EN→ES previa":         resumen_D(despues, TAU_ELEGIDO)}).T
tabla_D["¿alcanza el criterio 0,80?"] = ["sí" if v >= 0.80 else "no" for v in tabla_D["recall@5"]]

titulo("hipótesis H3 · efecto de traducir la consulta antes de recuperar")
display(tabla_D)
RESULTADOS["traduccion"] = {"modelo": MODELO_TRAD, "ms_por_consulta": round(t_trad, 1),
                            "tabla": tabla_D.reset_index().to_dict("records")}

In [ ]:
#@title 11.3 · Figura 7 · La consulta en inglés antes y después de traducir { display-mode: "form" }
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.9))

m = ["recall@1", "recall@5", "MRR@10"]
x = np.arange(len(m)); w = 0.36
v1 = [tabla_D.loc["Sin traducción (consulta en inglés)", k] for k in m]
v2 = [tabla_D.loc["Con traducción EN→ES previa", k] for k in m]
b1 = ax[0].bar(x - w/2, v1, w, color=GRIS, label="sin traducción")
b2 = ax[0].bar(x + w/2, v2, w, color=VERDE, label="con traducción EN→ES")
ax[0].bar_label(b1, fmt="%.3f", fontsize=8); ax[0].bar_label(b2, fmt="%.3f", fontsize=8)
ax[0].axhline(0.80, color=ROJO, ls="--", lw=1.2)
ax[0].text(2.4, 0.82, "criterio 0,80", color=ROJO, fontsize=8, ha="right")
ax[0].set_xticks(x); ax[0].set_xticklabels(m); ax[0].set_ylim(0, 1.15)
ax[0].set_title("Subconjunto D · métricas de recuperación")
ax[0].legend(frameon=False, fontsize=8, loc="upper left")

ax[1].hist([x["top1"] for x in antes], bins=20, color=GRIS, alpha=0.8, label="sin traducción")
ax[1].hist([x["top1"] for x in despues], bins=20, color=VERDE, alpha=0.7, label="con traducción")
ax[1].axvline(TAU_ELEGIDO, color=ROJO, ls="--", lw=1.3)
ax[1].annotate(f"τ = {TAU_ELEGIDO}", xy=(TAU_ELEGIDO, 0), xytext=(5, 30),
               textcoords="offset points", color=ROJO, fontsize=8, fontweight="bold")
ax[1].set_xlabel("similitud del mejor fragmento"); ax[1].set_ylabel("consultas")
ax[1].set_title("Cuántas consultas en inglés superan el umbral")
ax[1].legend(frameon=False, fontsize=8)

fig.suptitle("Figura 7. La traducción previa desplaza la consulta en inglés al régimen en que el sistema ya funciona",
             fontweight="bold", y=1.05)
plt.show()

In [ ]:
#@title 11.4 · Comprobación: la traducción no toca ninguna forma quechua { display-mode: "form" }
# La regla 2 del bloque es verificable, no declarativa: se comprueba que la forma quechua
# devuelta procede literalmente del fragmento recuperado y no de la salida del traductor.
titulo("verificación de la salvaguarda en el flujo con traducción")
comprobadas, incidencias = 0, []
for e in D[:12]:
    s = np.maximum(puntuar("hibrido", e["consulta"]), puntuar("hibrido", TRAD[e["id"]]))
    j = int(np.argmax(s))
    frag = FRAG[j]
    forma = e["forma_esperada"]
    en_fragmento = norm(forma) in norm(frag["texto"])
    en_traduccion = norm(forma) in norm(TRAD[e["id"]])
    comprobadas += 1
    if en_traduccion:
        incidencias.append(e["id"])
    marca = "✓" if (s[j] >= TAU_ELEGIDO and en_fragmento) else ("·" if s[j] < TAU_ELEGIDO else "!")
    print(f"  {marca} {e['consulta'][:52]:<54} S={s[j]:.3f}  ->  "
          f"{frag['texto'][:38]!r} [{frag['documento'][:22]}, p. {frag['pagina']}]")
print(f"\nConsultas comprobadas: {comprobadas}")
print(f"Formas quechuas presentes en la salida del traductor: {len(incidencias)} "
      f"(debe ser 0: el traductor opera solo sobre español e inglés)")
print("Toda forma quechua mostrada procede literalmente del fragmento recuperado, con su procedencia.")

**Cómo se lee.** Este bloque es el que cierra el único criterio de éxito que la prueba de concepto
había dejado incumplido, y conviene interpretarlo con precisión.

La traducción **no mejora el modelo**: lo que hace es llevar la consulta al idioma en el que el corpus
está escrito, de modo que el sistema opere en el régimen en el que ya se había demostrado eficaz. Por
eso el resultado del subconjunto D tiende a acercarse al de los subconjuntos en español y no a un valor
intermedio: una vez traducida, una consulta en inglés es, a efectos de recuperación, una consulta en
español.

Tres cautelas antes de dar el requisito por resuelto:

1. **El techo lo fija la traducción, no la recuperación.** Cuando el traductor elige un sinónimo que no
   es el que emplea el diccionario indexado, la consulta falla aunque el corpus contenga la entrada.
   Buscar con las dos versiones —la original y la traducida— limita ese daño, pero no lo elimina.
2. **El umbral debe recalibrarse.** τ se calibró sobre consultas en español; incorporar un paso de
   traducción cambia la distribución de las similitudes de entrada y obliga a repetir el barrido del
   bloque 9 con el conjunto completo.
3. **Falta el costo real en el dispositivo.** Aquí se mide con un traductor específico sobre una unidad
   de procesamiento gráfico. En el sistema final la traducción la realizará el modelo de lenguaje local,
   y su costo se suma al presupuesto de latencia de RNF-01. Debe medirse en el equipo del proyecto antes
   del cierre del sprint 2.

---
## Bloque 12 · Prototipo mínimo

**Qué pregunta responde.** ¿Cómo se comporta el sistema completo ante consultas formuladas libremente?

El prototipo reúne todo lo anterior en el flujo que el PMV1 debe promover a código de producción:
depuración, traducción cuando la consulta viene en inglés, recuperación híbrida, decisión frente al
umbral y salida con procedencia o declaración de ausencia. No incluye el modelo generador: en esta
prueba la respuesta es el fragmento literal, precisamente para que nada de lo que se muestre pueda
haber sido redactado por un modelo.

> **Una prueba instructiva.** Escriba en el formulario del bloque 12.2 la consulta
> «¿cómo se dice casa en quechua wanka?». El sistema **se abstiene**, pese a que el corpus contiene la
> entrada: en el texto extraído esa entrada aparece fusionada con la siguiente (`CASA: Wasi.CASADO:
> Walmiyuq.`), de modo que la similitud se reparte y queda por debajo del umbral. Es uno de los 26
> falsos negativos del punto de operación, y muestra con precisión qué se paga por la ausencia de
> falsos positivos y por dónde se recupera: mejorando la ingesta, no bajando el umbral.

In [ ]:
#@title 12.1 · El flujo completo en una función { display-mode: "form" }
def es_ingles(q):
    marcas = re.findall(r"\b(how|what|say|word|the|for|in|of|to|is|do|you)\b", q.lower())
    return len(marcas) >= 2

def responder(consulta, tau=None, traducir=True, verbose=True):
    tau = TAU_ELEGIDO if tau is None else tau
    traza_q = {"consulta": consulta, "traduccion": None}
    s = puntuar("hibrido", consulta, depurada=True)
    if traducir and es_ingles(consulta):
        try:
            tr = traducir_en_es(consulta)
            traza_q["traduccion"] = tr
            s = np.maximum(s, puntuar("hibrido", tr, depurada=True))
        except Exception:
            pass
    j = int(np.argmax(s)); frag = FRAG[j]; score = float(s[j])
    resp = {"similitud": round(score, 3), "umbral": tau, **traza_q}
    if score >= tau:
        resp.update({"decision": "responde", "fragmento": frag["texto"],
                     "documento": frag["documento"], "pagina": frag["pagina"]})
    else:
        resp.update({"decision": "se abstiene",
                     "mensaje": "No hay respaldo documental para esa consulta en el corpus indexado."})
    if verbose:
        print("-" * 92)
        print(f"CONSULTA     {consulta}")
        if traza_q["traduccion"]:
            print(f"TRADUCIDA    {traza_q['traduccion']}   (registrada en la trazabilidad)")
        print(f"SIMILITUD    {resp['similitud']}   ·   umbral {tau}")
        if resp["decision"] == "responde":
            print(f"RESPUESTA    {resp['fragmento']}")
            print(f"PROCEDENCIA  {resp['documento']}, página {resp['pagina']}")
        else:
            print(f"RESPUESTA    {resp['mensaje']}")
            print( "             No se genera ninguna forma lingüística.")
    return resp

titulo("prototipo mínimo · cuatro consultas")
for q in ["¿cómo se dice zorro en quechua wanka?",
          "necesito el término wanka para referirme a la lluvia",
          "how do you say water in Wanka Quechua?",
          "¿cómo se dice criptomoneda en quechua wanka?"]:
    responder(q)

In [ ]:
#@title 12.2 · Consulta libre { display-mode: "form" }
CONSULTA = "¿cómo se dice lluvia en quechua wanka?"  #@param {type:"string"}
TRADUCIR_SI_ESTA_EN_INGLES = True  #@param {type:"boolean"}
UMBRAL = 0.41  #@param {type:"slider", min:0.05, max:0.95, step:0.01}

_ = responder(CONSULTA, tau=UMBRAL, traducir=TRADUCIR_SI_ESTA_EN_INGLES)

---
## Bloque 13 · Reporte consolidado y decisión

**Qué hace.** Reúne todas las cifras calculadas en la sesión, las contrasta con los criterios de éxito
comprometidos **antes** de ejecutar el experimento y exporta el resultado a un archivo JSON que puede
adjuntarse al Documento 6 como evidencia reproducible.

In [ ]:
#@title 13.1 · Contraste con los criterios de éxito { display-mode: "form" }
h1_a = EXP["hibrido"]["metricas"]["A_lexico_es"]["recall@5"]
h1_b = EXP["hibrido"]["metricas"]["B_independiente"]["recall@5"]
h2   = BARRIDOS["hibrido"]["sin_fp"]
h3_sin = tabla_D.loc["Sin traducción (consulta en inglés)", "recall@5"]
h3_con = tabla_D.loc["Con traducción EN→ES previa", "recall@5"]

validacion = pd.DataFrame([
    {"Hipótesis": "H1 · recuperación", "Criterio": "recall@5 ≥ 0,80 en A y B",
     "Resultado": f"A = {h1_a:.3f} · B = {h1_b:.3f}",
     "Cumple": "sí" if min(h1_a, h1_b) >= 0.80 else "no"},
    {"Hipótesis": "H2 · abstención", "Criterio": "cero falsos positivos con recall ≥ 0,70",
     "Resultado": f"τ = {h2['umbral']} · FP = {int(h2['FP'])} · recall = {h2['recall']:.3f} · F1 = {h2['F1']:.3f}",
     "Cumple": "sí" if h2["FP"] == 0 and h2["recall"] >= 0.70 else "no"},
    {"Hipótesis": "H3 · consulta en inglés", "Criterio": "recall@5 ≥ 0,80 en D",
     "Resultado": f"sin traducción = {h3_sin:.3f} · con traducción = {h3_con:.3f}",
     "Cumple": "no sin traducción; " + ("sí con traducción previa" if h3_con >= 0.80 else "tampoco con traducción previa")},
    {"Hipótesis": "RNF-02 · latencia", "Criterio": "≤ 100 ms por consulta",
     "Resultado": f"{EXP['hibrido']['latencia_ms']} ms con la estrategia híbrida",
     "Cumple": "sí" if EXP["hibrido"]["latencia_ms"] <= 100 else "no"},
    {"Hipótesis": "RF-07 · trazabilidad", "Criterio": "toda respuesta con documento y página",
     "Resultado": f"{sum(1 for f in FRAG if f.get('documento') and f.get('pagina'))} de {len(FRAG)} fragmentos con procedencia completa",
     "Cumple": "sí" if all(f.get("documento") and f.get("pagina") for f in FRAG) else "no"},
]).set_index("Hipótesis")

titulo("validación frente a los criterios comprometidos")
display(validacion)
RESULTADOS["validacion"] = validacion.reset_index().to_dict("records")

In [ ]:
#@title 13.2 · Exportación de la evidencia { display-mode: "form" }
import datetime, json
RESULTADOS["ejecucion"] = {
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "modelo_denso": MODELO,
    "modelo_traduccion": MODELO_TRAD,
    "umbral_adoptado": TAU_ELEGIDO,
    "dispositivo": dispositivo,
}

def serializable(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    return str(o)

SALIDA = "resultados_cuaderno.json"
with open(SALIDA, "w", encoding="utf-8") as fh:
    json.dump(RESULTADOS, fh, ensure_ascii=False, indent=1, default=serializable)

titulo("evidencia exportada", SALIDA)
print(json.dumps({k: (v if not isinstance(v, (list, dict)) else f"<{type(v).__name__} con {len(v)} elementos>")
                  for k, v in RESULTADOS.items()}, ensure_ascii=False, indent=1))
try:
    from google.colab import files
    files.download(SALIDA)
except Exception:
    print(f"\nDescargue manualmente el archivo {SALIDA} desde el panel de archivos.")

---
## Decisión: **GO CON CONDICIONES**

La arquitectura de recuperación aumentada es técnicamente viable sobre el corpus documental disponible
de quechua wanka y admite una salvaguarda de abstención sin falsos positivos, de modo que el PMV1 puede
iniciarse. Las condiciones que acompañan a la decisión son las que este cuaderno permite justificar una
por una:

| Condición | Qué debe resolverse | Bloque que la sustenta |
|---|---|---|
| **C-1 · Multilingüismo** | Traducir la consulta al español antes de recuperar y medir la ganancia con este mismo arnés antes del cierre del sprint 2. | 11 |
| **C-2 · Estrategia de recuperación** | Adoptar la híbrida léxica como línea base y conservar la densa como componente complementario, no como sustituto. | 6, 7 |
| **C-3 · Segmentación** | Segmentar por tipo documental, con una entrada por fragmento en el material lexicográfico y el tipo como campo obligatorio del índice. | 10 |
| **C-4 · Umbral de abstención** | Fijar τ por ausencia de falsos positivos, versionarlo en configuración y recalibrarlo ante cualquier cambio de corpus, segmentación o codificador. | 9 |
| **C-5 · Calibración del codificador denso** | Si el PMV2 lo conserva para el móvil, decidir la abstención por la diferencia entre el primer y el segundo resultado, no por el valor absoluto. | 8 |
| **C-6 · Conjunto de evaluación** | Incorporar consultas formuladas por docentes de Educación Intercultural Bilingüe y reportar sus resultados por separado. | 3 |

#### Qué **no** demuestra este cuaderno

- **No mide la generación de la respuesta.** El prototipo devuelve el fragmento literal; el
  comportamiento del modelo de lenguaje ante ese fragmento se mide en el PMV1.
- **No mide el desempeño en uso real.** Las consultas de A, B y D derivan del mismo diccionario que se
  indexa, de modo que sus cifras son el límite superior del desempeño.
- **No cubre la analítica predictiva (RF-14 a RF-16) ni el despliegue móvil**, que dependen de
  artefactos que los incrementos PMV2 y PMV3 deben producir.
- **No sustituye la validación por hablantes de la comunidad.** El sistema es una herramienta de
  consulta sobre fuentes ya publicadas y validadas, y no se constituye en autoridad lingüística sobre la
  variedad wanka.